<a href="https://colab.research.google.com/github/Doris-QZ/Fine-Tuning-BERT-Large-with-QLoRA-for-Author-Identification/blob/main/2_QLoRA_r16a8_QKV_Author_Identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Introduction

This notebook is part of the project on fine-tuning the **BERT Large** model using **QLoRA** for the  [**Spooky Author Identification**](https://www.kaggle.com/competitions/spooky-author-identification) dataset from Kaggle.  

It presents the **second-best model**, which follows the same fine-tuning procedure as [`QLoRA_r8a4_AllLin_Author_Identification.ipynb`](https://github.com/Doris-QZ/Fine-Tuning-BERT-Large-with-QLoRA-for-Author-Identification/blob/main/1_QLoRA_r8a4_AllLin_Author_Identification.ipynb), but with the following **hyperparameter configuration**:

> r=16  
> a=8  
> lora_dropout=0.1  
> learning_rate=2.5e-4  
> lr_scheduler_type='cosine'  
> warmup_ratio=0.01   
> weight_decay=0.01   
> target_modules=['query', 'key', 'value']

<br>  

**Data Source**: Meg Risdal and Rachael Tatman. Spooky Author Identification. https://kaggle.com/competitions/spooky-author-identification, 2017. Kaggle.

**Connect to Kaggle and download the dataset**

In [ ]:
# Install Kaggle library
!pip install kaggle

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Make a directory for Kaggle
!mkdir ~/.kaggle

# Copy the kaggle.json file to the directory
!cp /content/drive/MyDrive/ColabNotebooks/Kaggle_API_Key/kaggle.json ~/.kaggle/

# Change the file permission to read/write to the owner only
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle competitions download spooky-author-identification

# Unzip the data
!unzip spooky-author-identification.zip
!unzip train.zip
!unzip test.zip

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft

In [3]:
import bitsandbytes
import transformers
import peft
import accelerate

print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"accelerate: {accelerate.__version__}")

bitsandbytes: 0.48.1
transformers: 4.57.0
peft: 0.17.1
accelerate: 1.10.1


In [4]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, BertForSequenceClassification, BitsAndBytesConfig
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from transformers import EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft import TaskType, replace_lora_weights_loftq
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

### Dataset Preparation

Since I already performed Exploratory Data Analysis in another [notebook](https://github.com/Doris-QZ/spooky_author_identification/blob/main/0_Hybrid_Spooky_Author_Identification.ipynb), I’ll skip the EDA section here and go straight to preparing the dataset for the model.

In [5]:
# Load the data
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

# Take a look at the data
train.info()
train.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19579 entries, 0 to 19578
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      19579 non-null  object
 1   text    19579 non-null  object
 2   author  19579 non-null  object
dtypes: object(3)
memory usage: 459.0+ KB


,id,text,author
0,id26305,"This process, however, afforded me no means of...",EAP
1,id17569,It never once occurred to me that the fumbling...,HPL
2,id11008,"In his left hand was a gold snuff box, from wh...",EAP
3,id27763,How lovely is spring As we looked from Windsor...,MWS
4,id12958,"Finding nothing else, not even gold, the Super...",HPL


In [6]:
# Create a new column of 'label' to encode 'author' to numeric
label_to_id = {'EAP': 0, 'MWS': 1, 'HPL': 2}
train['labels'] = train['author'].map(label_to_id)

In [7]:
# Split the 'train' data into training and validation dataset
train_set, val_set = train_test_split(train, test_size=0.1, random_state=42)

In [ ]:
# Load the pretrained tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-large-uncased")

In [9]:
# Tokenize text data
train_tokenized = tokenizer(train_set['text'].tolist(),
                            max_length=512,
                            truncation=True,
                            padding=False)

val_tokenized = tokenizer(val_set['text'].tolist(),
                          max_length=512,
                          truncation=True,
                          padding=False)

In [11]:
class TextDataset(torch.utils.data.Dataset):
    """
    Create a PyTorch Dataset object from tokenized text data and labels.

    Args:
        data (dict): Dictionary of lists from deberta-v3-large tokenizer(without padding)
        label (list): None or List of numerical labels

    Returns:
        dict[str, torch.tensor]: A dictionary containing a single sample's tensors.

    """
    def __init__(self, data, label=None):
        self.data = data
        self.label = label

    def __len__(self):
        return len(self.data['input_ids'])

    def __getitem__(self, idx):
        item = {'input_ids': torch.tensor(self.data['input_ids'][idx]),
                'token_type_ids': torch.tensor(self.data['token_type_ids'][idx]),
                'attention_mask': torch.tensor(self.data['attention_mask'][idx])}
        if self.label:
            item['labels'] = torch.tensor(self.label[idx])
        return item

In [12]:
train_dataset = TextDataset(train_tokenized, train_set['labels'].tolist())
val_dataset = TextDataset(val_tokenized, val_set['labels'].tolist())
train_dataset[0]

{'input_ids': tensor([  101,  2145,  2500,  1010,  2164,  3533,  2370,  1010,  2031,  8106,
          2205,  3748,  1998, 10392,  2005, 17358, 13675, 14728,  5897,  1012,
           102]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'labels': tensor(2)}

### Model Preparation

To fine-tune a large language model using QLoRA within the Hugging Face ecosystem, the process typically involves three main steps:
1. Set the quantization configuration -> Load the quantized model
2. Prepare the model for k-bit training
3. Set the LoRA configuration -> Load the PEFT model with the LoRA configuration

In [ ]:
# Set quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load the quantized model
model = BertForSequenceClassification.from_pretrained(
    "bert-large-uncased",
    num_labels=3,
    quantization_config=quantization_config,
    device_map='auto'
)

In [16]:
# Prepare the model for kbit training
model = prepare_model_for_kbit_training(model)
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (key): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (value): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (LayerNor

In [17]:
# Set LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=['query', 'key', 'value'],
    bias='none',
    task_type=TaskType.SEQ_CLS
)

# Load the PEFT model with the LoRA configuration
r16a8QKV = get_peft_model(model, lora_config)

In [18]:
# Reinitialize LoRA weights using LoftQ
replace_lora_weights_loftq(r16a8QKV)

In [19]:
# Unfreeze the classifier weights
for name, param in r16a8QKV.named_parameters():
    if 'classifier' in name:
        param.requires_grad = True

In [21]:
r16a8QKV.print_trainable_parameters()

trainable params: 2,365,446 || all params: 337,507,334 || trainable%: 0.7009


In [23]:
# Create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

# Define the metrics
def compute_metrics(eval_pred):
    y_pred, y_true = eval_pred
    y_pred = np.argmax(y_pred, axis=1)
    accuracy = accuracy_score(y_true, y_pred)
    return {'accuracy': accuracy}


In [24]:
# Set up the training arguments
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/ColabNotebooks/Spooky_Author_Identification/qlora/r16a8QKV_3',
    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=2.5e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.01,
    weight_decay=0.01,
    max_grad_norm=0.3,
    eval_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    gradient_checkpointing=True,
    report_to = "none"
)

In [25]:
# Set up the trainer
trainer = Trainer(
    model=r16a8QKV,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

r16a8QKV.config.use_cache = False

In [26]:
# Training
training_log = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.806600,0.595226,0.750766
2,0.474300,0.476296,0.810010
3,0.393500,0.426069,0.830439
4,0.341000,0.412013,0.846782
5,0.293300,0.402502,0.843207
6,0.257300,0.366123,0.858018
7,0.225100,0.402792,0.849847
8,0.188000,0.387240,0.867722
9,0.169800,0.450063,0.843207
